# Within-Dataset Benchmark

This notebook follows the benchmark structure from the IMPROVE cross-dataset DRP paper for the **within-dataset** setting:

1. Use the official IMPROVE benchmark data.
2. Use official split files: `<dataset>_split_<fold>_{train,val,test}.txt`.
3. Build a tabular ML matrix from cancer gene expression plus drug Mordred descriptors.
4. Fit preprocessing only on train entities.
5. Train multiple regressors and evaluate on the held-out test split from the same dataset.

Default config is a smoke test: `CCLE`, fold `0`. Expand `DATASETS` and `FOLDS` after the notebook runs end-to-end.

In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

RANDOM_STATE = 42
ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "csa_data" / "raw_data"
X_DIR = DATA_DIR / "x_data"
Y_DIR = DATA_DIR / "y_data"
SPLIT_DIR = DATA_DIR / "splits"
OUT_DIR = ROOT / "new_notebook" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (Y_DIR / "response.tsv").exists(), f"Missing response.tsv under {Y_DIR}"
assert SPLIT_DIR.exists(), f"Missing split dir: {SPLIT_DIR}"

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR : {OUT_DIR}")
print(f"LightGBM available: {HAS_LIGHTGBM}")

DATA_DIR: /Users/vietanh/Desktop/ML-Predicting-drug-response/data/csa_data/raw_data
OUT_DIR : /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results
LightGBM available: True


## Configuration

`TOP_GE_FEATURES` and `TOP_DRUG_FEATURES` keep the notebook practical for repeated model benchmarking. Set them to larger values if you have enough RAM/time.

In [2]:
# Smoke-test config. After this works, uncomment the full benchmark config.
DATASETS = ["CCLE"]
FOLDS = [0]

# Full within-dataset benchmark config.
# DATASETS = ["CCLE", "CTRPv2", "gCSI", "GDSCv1", "GDSCv2"]
# FOLDS = list(range(10))

TARGET_COL = "auc"
CELL_ID_COL = "improve_sample_id"
DRUG_ID_COL = "improve_chem_id"

TOP_GE_FEATURES = 512
TOP_DRUG_FEATURES = 512

# Subsample train rows for quick iteration. Set to None for final runs.
MAX_TRAIN_ROWS = None
MAX_EVAL_ROWS = None

print({
    "DATASETS": DATASETS,
    "FOLDS": FOLDS,
    "TOP_GE_FEATURES": TOP_GE_FEATURES,
    "TOP_DRUG_FEATURES": TOP_DRUG_FEATURES,
})

{'DATASETS': ['CCLE'], 'FOLDS': [0], 'TOP_GE_FEATURES': 512, 'TOP_DRUG_FEATURES': 512}


## Load Response And Feature Tables

In [3]:
response = pd.read_csv(Y_DIR / "response.tsv", sep="\t", low_memory=False)
response = response.reset_index().rename(columns={"index": "global_index"})
response[CELL_ID_COL] = response[CELL_ID_COL].astype(str)
response[DRUG_ID_COL] = response[DRUG_ID_COL].astype(str)

summary = response.groupby("source").agg(
    responses=(TARGET_COL, "size"),
    cells=(CELL_ID_COL, "nunique"),
    drugs=(DRUG_ID_COL, "nunique"),
    auc_mean=(TARGET_COL, "mean"),
    auc_std=(TARGET_COL, "std"),
)
display(summary.loc[[d for d in DATASETS if d in summary.index]])

def read_gene_expression(path):
    ge = pd.read_csv(path, sep="\t", skiprows=[1, 2], index_col=0, low_memory=False)
    ge.index = ge.index.astype(str)
    ge = ge[~ge.index.duplicated(keep="first")]
    return ge.apply(pd.to_numeric, errors="coerce")

def read_mordred(path):
    md = pd.read_csv(path, sep="\t", low_memory=False)
    md[DRUG_ID_COL] = md[DRUG_ID_COL].astype(str)
    md = md.drop_duplicates(DRUG_ID_COL).set_index(DRUG_ID_COL)
    return md.apply(pd.to_numeric, errors="coerce")

print("Loading gene expression...")
gene_expression = read_gene_expression(X_DIR / "cancer_gene_expression.tsv")
print("gene_expression:", gene_expression.shape)

print("Loading Mordred descriptors...")
mordred = read_mordred(X_DIR / "drug_mordred.tsv")
print("mordred:", mordred.shape)

,responses,cells,drugs,auc_mean,auc_std
source,,,,,
CCLE,9519,411,24,0.7856,0.1612


Loading gene expression...
gene_expression: (1007, 30805)
Loading Mordred descriptors...
mordred: (1565, 1613)


## Helpers

Feature selection is fit using the training split only. Imputers/scalers are also fit inside each model pipeline using train rows only.

In [4]:
def load_split_indices(dataset, fold, stage):
    path = SPLIT_DIR / f"{dataset}_split_{fold}_{stage}.txt"
    if not path.exists():
        raise FileNotFoundError(path)
    values = np.loadtxt(path, dtype=int)
    return np.atleast_1d(values)

def response_for_split(dataset, fold, stage):
    idx = load_split_indices(dataset, fold, stage)
    df = response.iloc[idx].copy()
    df = df[df["source"].eq(dataset)]
    df = df.dropna(subset=[TARGET_COL, CELL_ID_COL, DRUG_ID_COL])
    df[CELL_ID_COL] = df[CELL_ID_COL].astype(str)
    df[DRUG_ID_COL] = df[DRUG_ID_COL].astype(str)
    return df

def top_variance_columns(table, ids, top_k):
    matched = table.loc[table.index.intersection(pd.Index(ids))]
    if matched.empty:
        raise ValueError("No matching feature rows for train IDs")
    variances = matched.var(axis=0, skipna=True).fillna(0)
    variances = variances[variances > 0]
    if top_k is None or top_k >= len(variances):
        return variances.sort_values(ascending=False).index.tolist()
    return variances.nlargest(top_k).index.tolist()

def build_design_matrix(split_df, ge_cols, md_cols):
    ids = split_df[[CELL_ID_COL, DRUG_ID_COL, TARGET_COL]].copy()
    ids["row_id"] = np.arange(len(ids))

    ge_part = gene_expression[ge_cols].copy()
    ge_part.columns = [f"ge.{c}" for c in ge_part.columns]
    ge_part = ge_part.reset_index().rename(columns={ge_part.index.name or "index": CELL_ID_COL})

    md_part = mordred[md_cols].copy()
    md_part.columns = [f"mordred.{c}" for c in md_part.columns]
    md_part = md_part.reset_index().rename(columns={md_part.index.name or "index": DRUG_ID_COL})

    data = ids.merge(ge_part, on=CELL_ID_COL, how="inner")
    data = data.merge(md_part, on=DRUG_ID_COL, how="inner")
    data = data.sort_values("row_id").reset_index(drop=True)

    feature_cols = [c for c in data.columns if c.startswith("ge.") or c.startswith("mordred.")]
    X = data[feature_cols]
    y = data[TARGET_COL].astype(float).to_numpy()
    meta = data[[CELL_ID_COL, DRUG_ID_COL, TARGET_COL]].copy()
    return X, y, meta

def maybe_sample(X, y, meta, max_rows, seed):
    if max_rows is None or len(X) <= max_rows:
        return X, y, meta
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(X), size=max_rows, replace=False))
    return X.iloc[idx].reset_index(drop=True), y[idx], meta.iloc[idx].reset_index(drop=True)

def regression_metrics(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan
    return {"n": len(y_true), "rmse": rmse, "mae": mae, "r2": r2, "pearson": pearson}

## Models

In [5]:
models = {
    "ridge": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=10.0, random_state=RANDOM_STATE),
    ),
    "hist_gradient_boosting": make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=RANDOM_STATE),
    ),
    "random_forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
    ),
    "extra_trees": make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesRegressor(n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
    ),
}

if HAS_LIGHTGBM:
    models["lightgbm"] = make_pipeline(
        SimpleImputer(strategy="median"),
        LGBMRegressor(
            objective="regression",
            n_estimators=800,
            learning_rate=0.05,
            num_leaves=31,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=-1,
        ),
    )

list(models)

['ridge', 'hist_gradient_boosting', 'random_forest', 'extra_trees', 'lightgbm']

## Run Within-Dataset Benchmark

In [6]:
all_results = []

for dataset in DATASETS:
    for fold in FOLDS:
        print("=" * 80)
        print(f"Dataset={dataset} | fold={fold}")

        train_df = response_for_split(dataset, fold, "train")
        val_df = response_for_split(dataset, fold, "val")
        test_df = response_for_split(dataset, fold, "test")

        train_cells = train_df[CELL_ID_COL].unique()
        train_drugs = train_df[DRUG_ID_COL].unique()
        ge_cols = top_variance_columns(gene_expression, train_cells, TOP_GE_FEATURES)
        md_cols = top_variance_columns(mordred, train_drugs, TOP_DRUG_FEATURES)

        print(f"Rows raw: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")
        print(f"Features selected: ge={len(ge_cols):,}, mordred={len(md_cols):,}")

        X_train, y_train, meta_train = build_design_matrix(train_df, ge_cols, md_cols)
        X_val, y_val, meta_val = build_design_matrix(val_df, ge_cols, md_cols)
        X_test, y_test, meta_test = build_design_matrix(test_df, ge_cols, md_cols)

        X_train, y_train, meta_train = maybe_sample(X_train, y_train, meta_train, MAX_TRAIN_ROWS, RANDOM_STATE)
        X_val, y_val, meta_val = maybe_sample(X_val, y_val, meta_val, MAX_EVAL_ROWS, RANDOM_STATE)
        X_test, y_test, meta_test = maybe_sample(X_test, y_test, meta_test, MAX_EVAL_ROWS, RANDOM_STATE)

        print(f"Rows usable: train={len(X_train):,}, val={len(X_val):,}, test={len(X_test):,}")

        for model_name, estimator in models.items():
            print(f"  Training {model_name}...")
            start = time.time()
            model = clone(estimator)
            model.fit(X_train, y_train)
            train_seconds = time.time() - start

            for stage, X_stage, y_stage in [
                ("val", X_val, y_val),
                ("test", X_test, y_test),
            ]:
                pred = model.predict(X_stage)
                scores = regression_metrics(y_stage, pred)
                row = {
                    "analysis": "within_dataset",
                    "dataset": dataset,
                    "fold": fold,
                    "stage": stage,
                    "model": model_name,
                    "train_seconds": train_seconds,
                    "n_train": len(X_train),
                    "n_features": X_train.shape[1],
                    **scores,
                }
                all_results.append(row)
                print(
                    f"    {stage}: n={scores['n']:,} "
                    f"RMSE={scores['rmse']:.4f} MAE={scores['mae']:.4f} "
                    f"R2={scores['r2']:.4f} Pearson={scores['pearson']:.4f}"
                )

results = pd.DataFrame(all_results)
results_path = OUT_DIR / "within_dataset_results.csv"
results.to_csv(results_path, index=False)
print(f"Saved: {results_path}")
display(results)

Dataset=CCLE | fold=0
Rows raw: train=7,616, val=952, test=951
Features selected: ge=512, mordred=512
Rows usable: train=7,616, val=952, test=951
  Training ridge...
    val: n=952 RMSE=0.0821 MAE=0.0630 R2=0.7174 Pearson=0.8473
    test: n=951 RMSE=0.0843 MAE=0.0655 R2=0.7388 Pearson=0.8596
  Training hist_gradient_boosting...
    val: n=952 RMSE=0.0715 MAE=0.0556 R2=0.7857 Pearson=0.8867
    test: n=951 RMSE=0.0741 MAE=0.0582 R2=0.7981 Pearson=0.8942
  Training random_forest...
    val: n=952 RMSE=0.0798 MAE=0.0621 R2=0.7332 Pearson=0.8564
    test: n=951 RMSE=0.0829 MAE=0.0642 R2=0.7473 Pearson=0.8646
  Training extra_trees...
    val: n=952 RMSE=0.0735 MAE=0.0580 R2=0.7737 Pearson=0.8798
    test: n=951 RMSE=0.0779 MAE=0.0603 R2=0.7774 Pearson=0.8819
  Training lightgbm...
    val: n=952 RMSE=0.0694 MAE=0.0543 R2=0.7979 Pearson=0.8936
    test: n=951 RMSE=0.0719 MAE=0.0558 R2=0.8099 Pearson=0.9005
Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/within

/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,analysis,dataset,fold,stage,model,train_seconds,n_train,n_features,n,rmse,mae,r2,pearson
0,within_dataset,CCLE,0,val,ridge,0.5087,7616,1024,952,0.0821,0.0630,0.7174,0.8473
1,within_dataset,CCLE,0,test,ridge,0.5087,7616,1024,951,0.0843,0.0655,0.7388,0.8596
2,within_dataset,CCLE,0,val,hist_gradient_boosting,11.5128,7616,1024,952,0.0715,0.0556,0.7857,0.8867
3,within_dataset,CCLE,0,test,hist_gradient_boosting,11.5128,7616,1024,951,0.0741,0.0582,0.7981,0.8942
4,within_dataset,CCLE,0,val,random_forest,60.6389,7616,1024,952,0.0798,0.0621,0.7332,0.8564
5,within_dataset,CCLE,0,test,random_forest,60.6389,7616,1024,951,0.0829,0.0642,0.7473,0.8646
6,within_dataset,CCLE,0,val,extra_trees,30.1494,7616,1024,952,0.0735,0.0580,0.7737,0.8798
7,within_dataset,CCLE,0,test,extra_trees,30.1494,7616,1024,951,0.0779,0.0603,0.7774,0.8819
8,within_dataset,CCLE,0,val,lightgbm,12.6272,7616,1024,952,0.0694,0.0543,0.7979,0.8936
9,within_dataset,CCLE,0,test,lightgbm,12.6272,7616,1024,951,0.0719,0.0558,0.8099,0.9005


## Summary Tables

In [7]:
if not results.empty:
    test_results = results[results["stage"].eq("test")]
    summary = (
        test_results
        .groupby(["dataset", "model"])
        .agg(
            folds=("fold", "nunique"),
            rmse_mean=("rmse", "mean"),
            rmse_std=("rmse", "std"),
            mae_mean=("mae", "mean"),
            r2_mean=("r2", "mean"),
            pearson_mean=("pearson", "mean"),
            train_seconds_mean=("train_seconds", "mean"),
        )
        .reset_index()
        .sort_values(["dataset", "rmse_mean"])
    )
    summary_path = OUT_DIR / "within_dataset_summary.csv"
    summary.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")
    display(summary)

Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/within_dataset_summary.csv


,dataset,model,folds,rmse_mean,rmse_std,mae_mean,r2_mean,pearson_mean,train_seconds_mean
2,CCLE,lightgbm,1,0.0719,NaN,0.0558,0.8099,0.9005,12.6272
1,CCLE,hist_gradient_boosting,1,0.0741,NaN,0.0582,0.7981,0.8942,11.5128
0,CCLE,extra_trees,1,0.0779,NaN,0.0603,0.7774,0.8819,30.1494
3,CCLE,random_forest,1,0.0829,NaN,0.0642,0.7473,0.8646,60.6389
4,CCLE,ridge,1,0.0843,NaN,0.0655,0.7388,0.8596,0.5087


## Next Step: Cross-Dataset

For cross-dataset benchmarking, reuse the same trained source-dataset model and evaluate on target datasets. The important change is feature handling: feature selectors, imputers, and scalers must be fit on the source train split only, then applied to target response rows.